<a href="https://colab.research.google.com/github/ClaymentPW/Tugas-Overclock-2026/blob/main/Machine_Learning_OC_i_Smile_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display, Markdown
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
N_REPEATS = 10
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Klasifikasi Kualitas Susu — Evaluasi SVM (I-Smile OC26)

**Tujuan:** memprediksi kualitas susu (*high, low, medium*) menggunakan fitur `pH`, `Temprature`, `Taste`, `Odor`, `Fat`, `Turbidity`, dan `Colour`.

**Desain evaluasi:**
1. Identifikasi duplikat berdasarkan **seluruh kombinasi fitur** sebagai `groups`. Gunakan *StratifiedGroupKFold* untuk menghindari kombinasi fitur yang sama muncul di training dan testing.
2. Pisahkan **holdout test** satu kali menggunakan `RANDOM_STATE = 42`, lalu **jangan gunakan holdout untuk memilih model**.
3. Jalankan GridSearchCV hanya pada **training set**. Bandingkan baseline dan tuned dengan *repeated stratified grouped cross-validation* **hanya pada training set** (5 fold × 10 seed), memakai **F1-macro** sebagai metrik utama.
4. Pilih model berdasarkan rata-rata F1-macro CV training, fit pada seluruh training, lalu evaluasi sekali pada holdout test. Tampilkan Accuracy, Precision, Recall, F1-score, serta confusion matrix.
5. Confusion matrix CV dibuat dari **out-of-fold predictions training** (satu 5-fold CV dengan seed tetap). Ini bukan matriks holdout test dan bukan akumulasi 10 pengulangan. Kedua matriks disajikan terpisah.

**Catatan:** dataset ini memuat banyak pengulangan kombinasi fitur sehingga 100% pada satu split test bukan jaminan model akan selalu sempurna. Hasil CV merupakan pembanding utama *di antara kandidat model*, sedangkan hasil holdout merupakan evaluasi akhir pada data yang disisihkan. CV setelah tuning tetap memiliki potensi bias seleksi karena pemilihan parameter awal terjadi pada training yang sama; evaluasi nested CV diperlukan untuk estimasi generalisasi yang lebih ketat.

**Cara menjalankan:** letakkan `milknew.csv` di folder notebook, atau jalankan di Google Colab dan unggah CSV ketika diminta. Jalankan semua sel berurutan. File `README.md`, `documentation.md`, dan artefak evaluasi akan dibuat otomatis di folder `output/` setelah selesai.

In [ ]:
CSV_PATH = 'milknew.csv'

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print('Unggah dataset milknew.csv dari Milk Quality Dataset (Kaggle).')
        uploaded = files.upload()
        if 'milknew.csv' in uploaded:
            CSV_PATH = 'milknew.csv'
        elif len(uploaded) == 1 and next(iter(uploaded)).lower().endswith('.csv'):
            CSV_PATH = next(iter(uploaded))
        else:
            raise FileNotFoundError('Unggah tepat satu file CSV dataset Milk Quality.')
    except ImportError:
        raise FileNotFoundError(
            'Dataset milknew.csv belum tersedia. Letakkan di direktori notebook lalu Run All. '
            'Sumber: https://www.kaggle.com/datasets/cpluzshrijayan/milkquality'
        )

df = pd.read_csv(CSV_PATH)
print('Ukuran dataset:', df.shape)
display(df.head())

Unggah dataset milknew.csv dari Milk Quality Dataset (Kaggle).


In [ ]:
df.columns = df.columns.str.strip()
required = {'pH','Temprature','Taste','Odor','Fat','Turbidity','Colour','Grade'}
assert required.issubset(df.columns), f'Kolom hilang: {required - set(df.columns)}'
assert not df[list(required)].isna().any().any(), 'Ditemukan missing value: periksa dataset sebelum melanjutkan.'
df['Grade'] = (df['Grade'].astype(str).str.strip().str.lower()
               .str.replace('_quality','', regex=False))

le = LabelEncoder()
df['Grade_encoded'] = le.fit_transform(df['Grade'])
assert len(le.classes_) == 3, f'Harus ada 3 kelas, ditemukan: {le.classes_}'

X = df.drop(columns=['Grade','Grade_encoded'])
y = df['Grade_encoded']
groups = X.astype(str).agg('|'.join, axis=1)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]
assert set(g_train).isdisjoint(set(g_test)), 'Duplikat fitur menyeberang train-test.'
print('Label:', dict(zip(le.classes_, le.transform(le.classes_))))
print(f'Train: {len(X_train)} baris ({g_train.nunique()} kombinasi unik)')
print(f'Test : {len(X_test)} baris ({g_test.nunique()} kombinasi unik)')
print('Duplikat kombinasi fitur:', int(groups.duplicated().sum()))
print('Tidak ada grup fitur train yang muncul di test: BENAR')

Label: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}
Train: 904 baris (69 kombinasi unik)
Test : 155 baris (14 kombinasi unik)
Duplikat kombinasi fitur: 976
Tidak ada grup fitur train yang muncul di test: BENAR


In [ ]:
NUM_COLS = ['pH', 'Temprature', 'Colour']
BIN_COLS = ['Taste', 'Odor', 'Fat', 'Turbidity']
assert set(NUM_COLS + BIN_COLS) == set(X.columns), 'Ada fitur tak terduga.'

def make_svm(**params):
    return Pipeline([
        ('prep', ColumnTransformer([('scale', StandardScaler(), NUM_COLS)], remainder='passthrough')),
        ('svc', SVC(random_state=RANDOM_STATE, **params))
    ])

In [ ]:
baseline = make_svm()
baseline.fit(X_train, y_train)
train_pred_baseline = baseline.predict(X_train)
print('Baseline — performa training (diagnostik; bukan estimasi generalisasi)')
print('Accuracy:', f'{accuracy_score(y_train, train_pred_baseline):.4f}')
print('F1-macro:', f'{f1_score(y_train, train_pred_baseline, average="macro"):.4f}')

In [ ]:
unique_group_class_counts = (pd.DataFrame({'group':g_train.values, 'class':y_train.values})
                             .drop_duplicates()['class'].value_counts())
n_splits = int(min(5, unique_group_class_counts.min()))
if n_splits < 2:
    raise ValueError('Jumlah grup per kelas tidak cukup untuk grouped cross-validation.')
print('Grup unik tiap kelas (training):', unique_group_class_counts.sort_index().to_dict())
print('Fold CV:', n_splits)

param_grid = {
    'svc__kernel': ['linear','rbf'],
    'svc__C': [0.1,1,10,100],
    'svc__gamma': ['scale',0.01,0.1,1]
}
tuning_cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(make_svm(), param_grid, scoring='f1_macro', cv=tuning_cv,
                    n_jobs=-1, return_train_score=True, error_score='raise')
grid.fit(X_train, y_train, groups=g_train)
best_params = {key.replace('svc__',''):value for key,value in grid.best_params_.items()}
print('Parameter terbaik GridSearchCV:', best_params)
print('F1-macro CV saat tuning:', f'{grid.best_score_:.4f}')

In [ ]:
res = pd.DataFrame(grid.cv_results_)
cols = ["param_svc__kernel", "param_svc__C", "param_svc__gamma",
        "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]
res_tbl = (res[cols]
           .rename(columns={"param_svc__kernel": "kernel", "param_svc__C": "C",
                            "param_svc__gamma": "gamma", "mean_train_score": "F1 train (CV)",
                            "mean_test_score": "F1 validasi (CV)", "std_test_score": "std",
                            "rank_test_score": "rank"})
           .sort_values("rank").reset_index(drop=True))
res_tbl.head(10).round(4)

In [ ]:
rbf = res[res["param_svc__kernel"] == "rbf"].copy()
rbf["gamma"] = rbf["param_svc__gamma"].astype(str)
rbf["C"] = rbf["param_svc__C"].astype(float)
pivot = rbf.pivot_table(index="C", columns="gamma", values="mean_test_score")

plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("F1-macro validasi (CV) — SVM kernel RBF")
plt.xlabel("gamma"); plt.ylabel("C")
plt.tight_layout(); plt.show()

In [ ]:
def repeated_grouped_cv_scores(estimator, features, labels, grp, repeats=N_REPEATS):
    scores = []
    for seed in range(repeats):
        splitter_cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        scores.append(cross_val_score(estimator, features, labels, groups=grp,
                                      cv=splitter_cv, scoring='f1_macro', n_jobs=-1))
    return np.asarray(scores)
baseline_cv_per_seed = repeated_grouped_cv_scores(make_svm(), X_train, y_train, g_train)
tuned_cv_per_seed = repeated_grouped_cv_scores(make_svm(**best_params), X_train, y_train, g_train)
baseline_mean_per_seed = baseline_cv_per_seed.mean(axis=1)
tuned_mean_per_seed = tuned_cv_per_seed.mean(axis=1)
diff_per_seed = tuned_mean_per_seed - baseline_mean_per_seed
wins_baseline = int((baseline_mean_per_seed > tuned_mean_per_seed).sum())
wins_tuned = int((tuned_mean_per_seed > baseline_mean_per_seed).sum())
ties = N_REPEATS - wins_baseline - wins_tuned

cv_baseline_mean, cv_baseline_std = baseline_cv_per_seed.mean(), baseline_cv_per_seed.std()
cv_tuned_mean, cv_tuned_std = tuned_cv_per_seed.mean(), tuned_cv_per_seed.std()
print(f'Repeated grouped {n_splits}-fold × {N_REPEATS} pengulangan (TRAINING SAJA)')
print(f'Baseline F1-macro : {cv_baseline_mean:.4f} ± {cv_baseline_std:.4f}')
print(f'Tuned F1-macro    : {cv_tuned_mean:.4f} ± {cv_tuned_std:.4f}')
print(f'Perbedaan tuned-baseline (rata-rata per seed): {diff_per_seed.mean():+.4f}')
print(f'Menang baseline: {wins_baseline}/{N_REPEATS}; tuned: {wins_tuned}/{N_REPEATS}; seri: {ties}/{N_REPEATS}')

In [ ]:
chosen_name = 'Baseline' if cv_baseline_mean >= cv_tuned_mean else 'Tuned'
chosen_params = {} if chosen_name == 'Baseline' else best_params
best_model = make_svm(**chosen_params)
best_model.fit(X_train, y_train)
y_pred_final = best_model.predict(X_test)
y_pred_baseline_test = baseline.predict(X_test)

comparison = pd.DataFrame({
    'Model': ['Baseline','Tuned'],
    'CV F1-macro (mean ± std, training)': [
        f'{cv_baseline_mean:.4f} ± {cv_baseline_std:.4f}',
        f'{cv_tuned_mean:.4f} ± {cv_tuned_std:.4f}'
    ],
    'Menang per seed': [wins_baseline, wins_tuned],
    'Status': ['Dipilih' if chosen_name=='Baseline' else 'Alternatif',
               'Dipilih' if chosen_name=='Tuned' else 'Alternatif']
})
display(comparison)
print(f'Model final: {chosen_name} SVM. Hasil holdout dievaluasi berikutnya, bukan untuk seleksi.')

In [ ]:
labels = np.arange(len(le.classes_))
holdout_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_final),
    'Precision macro': precision_score(y_test, y_pred_final, average='macro', zero_division=0),
    'Recall macro': recall_score(y_test, y_pred_final, average='macro', zero_division=0),
    'F1 macro': f1_score(y_test, y_pred_final, average='macro', zero_division=0)
}
print('=== METRIK HOLDOUT TEST (SATU SPLIT; BUKAN METRIK UTAMA PEMILIHAN) ===')
print(f'Sampel: {len(y_test)}, kombinasi fitur unik: {g_test.nunique()}')
for metric, value in holdout_metrics.items():
    print(f'{metric:18s}: {value:.4f}')
print('\nClassification report (per kelas):')
print(classification_report(y_test, y_pred_final, labels=labels,
                            target_names=le.classes_, zero_division=0))

cm_test = confusion_matrix(y_test, y_pred_final, labels=labels)
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm_test, display_labels=le.classes_).plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
ax.grid(False, which="both", axis="both")
ax.xaxis.grid(False, which="both")
ax.yaxis.grid(False, which="both")
ax.set_title(f'Confusion Matrix HOLDOUT — {chosen_name} SVM (n={len(y_test)})')
plt.tight_layout()
fig.savefig(OUTPUT_DIR/'confusion_matrix_holdout.png', dpi=160, bbox_inches='tight')
plt.show()

## Confusion matrix *cross-validation* (out-of-fold)

Setiap sampel **training** diprediksi tepat satu kali ketika berada di fold validasi (*out-of-fold*, OOF) dalam **satu pembagian CV** dengan seed `42`. Oleh karena itu, jumlah seluruh sel pada confusion matrix CV = jumlah baris training, bukan ukuran holdout, dan bukan `10 ×` jumlah training. Model di-fit ulang pada fold training dengan pipeline yang sama. Confusion matrix ini melengkapi rata-rata F1-macro dari 10 pengulangan, **tidak menggantikannya**.


In [ ]:
oof_cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
oof_pred = cross_val_predict(make_svm(**chosen_params), X_train, y_train,
                             groups=g_train, cv=oof_cv, n_jobs=-1)
cm_oof = confusion_matrix(y_train, oof_pred, labels=labels)
assert cm_oof.sum() == len(y_train), 'Jumlah OOF harus sama dengan training.'
oof_metrics = {
    'Accuracy': accuracy_score(y_train, oof_pred),
    'Precision macro': precision_score(y_train, oof_pred, average='macro', zero_division=0),
    'Recall macro': recall_score(y_train, oof_pred, average='macro', zero_division=0),
    'F1 macro': f1_score(y_train, oof_pred, average='macro', zero_division=0)
}
print('=== OOF CV — satu grouped CV (TRAINING) ===')
print(f'Sampel OOF: {len(oof_pred)}; kombinasi fitur unik: {g_train.nunique()}')
for metric, value in oof_metrics.items():
    print(f'{metric:18s}: {value:.4f}')
print(classification_report(y_train, oof_pred, labels=labels,
                            target_names=le.classes_, zero_division=0))
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm_oof, display_labels=le.classes_).plot(ax=ax, cmap='Purples', values_format='d', colorbar=False)

ax.grid(False, which="both", axis="both")
ax.xaxis.grid(False, which="both")
ax.yaxis.grid(False, which="both")
ax.set_title(f'Confusion Matrix OOF grouped {n_splits}-fold — {chosen_name} (n={len(y_train)})')
plt.tight_layout()
fig.savefig(OUTPUT_DIR/'confusion_matrix_cv_oof.png', dpi=160, bbox_inches='tight')
plt.show()

## Interpretasi evaluasi (dihitung otomatis)

Teks interpretasi di bawah bersumber langsung dari variabel hasil eksekusi, sehingga **tidak ada nilai CV, jumlah seed, atau ukuran testing yang ditulis manual**. Angka 1,0000 pada satu test split tidak otomatis membuktikan generalisasi sempurna. Cermati juga nilai rata-rata dan sebaran repeated grouped CV.


In [ ]:
test_correct = int(np.trace(cm_test))
test_total = int(cm_test.sum())
main_mean = cv_baseline_mean if chosen_name=='Baseline' else cv_tuned_mean
main_std = cv_baseline_std if chosen_name=='Baseline' else cv_tuned_std
cv_delta = cv_tuned_mean - cv_baseline_mean

interp = [
    f'**Accuracy holdout = {holdout_metrics["Accuracy"]:.4f}**: {test_correct} dari {test_total} sampel test diklasifikasikan benar. '
    f'Test hanya memuat {g_test.nunique()} kombinasi fitur unik, sehingga satu nilai holdout memiliki ketidakpastian tinggi.',
    f'**Precision macro = {holdout_metrics["Precision macro"]:.4f}**: rata-rata precision antar kelas (tanpa bobot jumlah sampel); '
    'mengukur ketepatan label yang diprediksi untuk setiap kelas.',
    f'**Recall macro = {holdout_metrics["Recall macro"]:.4f}**: rata-rata kemampuan model menemukan setiap kelas yang sebenarnya.',
    f'**F1-macro holdout = {holdout_metrics["F1 macro"]:.4f}**: rata-rata harmonik precision dan recall per kelas, '
    'memperlakukan setiap kelas dengan bobot setara.',
    f'**Estimasi perbandingan utama (CV training)**: model terpilih {chosen_name} meraih F1-macro '
    f'{main_mean:.4f} ± {main_std:.4f} pada {n_splits} fold × {N_REPEATS} seed. '
    'Angka ± menunjukkan simpangan baku antar skor fold (bukan confidence interval).',
    f'**Perbandingan baseline vs tuned**: baseline {cv_baseline_mean:.4f} ± {cv_baseline_std:.4f}, '
    f'tuned {cv_tuned_mean:.4f} ± {cv_tuned_std:.4f}; beda tuned − baseline {cv_delta:+.4f}. '
    f'Baseline unggul {wins_baseline} seed, tuned unggul {wins_tuned} seed, dan seri {ties} seed. '
    'Perbedaan ini tidak membuktikan penyebabnya overfitting, tetapi mendukung pilihan model berdasarkan protokol CV yang ditetapkan.',
    f'**Confusion matrix OOF**: menilai {len(y_train)} sampel training pada satu kali {n_splits}-fold grouped CV '
    f'(F1-macro OOF {oof_metrics["F1 macro"]:.4f}). '
    f'**Confusion matrix holdout** menilai {test_total} sampel test terpisah. '
    'Jangan menyamakan OOF satu seed dengan rata-rata 10 seed.'
]
display(Markdown('### Interpretasi hasil model\n\n' + '\n\n'.join('- ' + p for p in interp)))

## Output dan dokumentasi otomatis

Setelah sel berikut dijalankan, folder `output/` memuat prediksi holdout, prediksi OOF CV, confusion matrix, artefak model/encoder, parameter tuning, tabel CV, `README.md`, dan `documentation.md`. Nilai ringkasan **dihasilkan ulang setiap Run All** sehingga sesuai dengan hasil notebook pada eksekusi tersebut.


In [ ]:
joblib.dump(best_model, OUTPUT_DIR/'svm_milk_quality.joblib')
joblib.dump(le, OUTPUT_DIR/'label_encoder.joblib')

prediksi_test = X_test.copy()
prediksi_test['Grade_aktual'] = le.inverse_transform(y_test.to_numpy())
prediksi_test['Grade_prediksi'] = le.inverse_transform(y_pred_final)
prediksi_test.to_csv(OUTPUT_DIR/'prediksi_test_svm.csv', index=False)

prediksi_oof = X_train.copy()
prediksi_oof['Grade_aktual'] = le.inverse_transform(y_train.to_numpy())
prediksi_oof['Grade_prediksi_oof'] = le.inverse_transform(oof_pred)
prediksi_oof.to_csv(OUTPUT_DIR/'prediksi_cv_oof_svm.csv', index=False)
comparison.to_csv(OUTPUT_DIR/'perbandingan_cv.csv', index=False)
pd.DataFrame(grid.cv_results_).to_csv(OUTPUT_DIR/'cv_results_tuning.csv', index=False)

with open(OUTPUT_DIR/'best_params.json','w', encoding='utf-8') as f:
    json.dump({'tuning_best_params':best_params, 'final_model':chosen_name,
               'selected_model_params':chosen_params, 'random_state':RANDOM_STATE,
               'n_splits':n_splits, 'n_repeats':N_REPEATS}, f, indent=2)
with open(OUTPUT_DIR/'split_indices.json','w') as f:
    json.dump({'train_idx':train_idx.tolist(), 'test_idx':test_idx.tolist()}, f)

quality_notes = [
    f'Dataset terdiri dari {len(df)} baris dan {groups.nunique()} kombinasi fitur unik.',
    f'Holdout {len(y_test)} sampel ({g_test.nunique()} kombinasi unik) independen berdasarkan grup dari training {len(y_train)} sampel.',
    'Data dengan kombinasi fitur identik tidak boleh tersebar antar training dan testing.',
    'GridSearchCV dan seleksi model melalui repeated grouped CV dijalankan pada training saja.',
    'Standarisasi fitur numerik dilakukan dalam pipeline untuk menghindari leakage antar fold.',
    'Nilai holdout sempurna pada subset kecil tidak berarti akurasi populasi pasti sempurna.',
    'Repeated CV ini untuk membandingkan model di training; mean±std fold bukan confidence interval dan bukan nested CV.'
]
summary = '\n'.join([
    '# Dokumentasi Evaluasi — SVM Kualitas Susu',
    '',
    '## 1. Tujuan dan data',
    'Mengklasifikasikan kualitas susu menjadi high, low, dan medium dari tujuh fitur pengukuran.',
    'Sumber dataset: https://www.kaggle.com/datasets/cpluzshrijayan/milkquality',
    *('- ' + q for q in quality_notes[:3]),
    '',
    '## 2. Metode dan preprocessing',
    'Target dikodekan dengan LabelEncoder; pH, Temprature dan Colour distandardisasi dalam Pipeline; '
    'empat fitur biner diteruskan apa adanya. Split dan CV menggunakan StratifiedGroupKFold pada kombinasi fitur identik.',
    'GridSearchCV digunakan untuk mengeksplorasi parameter SVM hanya di training.',
    f'GridSearchCV memilih {best_params} dengan F1-macro {grid.best_score_:.4f}, tetapi model final dipilih '
    'berdasarkan rata-rata repeated CV training antara baseline dan tuned.',
    '',
    '## 3. Evaluasi repeated grouped cross-validation (training; utama untuk perbandingan model)',
    f'- Desain: {n_splits} fold × {N_REPEATS} seed; F1-macro.',
    f'- Baseline: {cv_baseline_mean:.4f} ± {cv_baseline_std:.4f}.',
    f'- Tuned: {cv_tuned_mean:.4f} ± {cv_tuned_std:.4f}.',
    f'- Baseline menang {wins_baseline}/{N_REPEATS} seed; tuned {wins_tuned}/{N_REPEATS}; seri {ties}/{N_REPEATS}.',
    f'- Model final: **{chosen_name} SVM**.',
    '',
    '## 4. Evaluasi holdout test (satu split; setelah seleksi model)',
    f'- N = {test_total} sampel; {g_test.nunique()} kombinasi fitur unik; benar {test_correct} sampel.',
    *[f'- {metric}: {value:.4f}.' for metric,value in holdout_metrics.items()],
    '',
    '## 5. Confusion matrix dan interpretasi',
    '- `confusion_matrix_cv_oof.png` = prediksi out-of-fold di training dalam satu CV seed 42.',
    '- `confusion_matrix_holdout.png` = prediksi holdout test satu split.',
    *('- ' + item.replace('**','') for item in interp),
    '',
    '## 6. Kesimpulan dan keterbatasan',
    f'Model {chosen_name} dipilih berdasarkan repeated CV training, bukan skor test.',
    'Repeated CV memperlihatkan variasi performa antar pembagian grup. Pengujian pada data baru dari sumber lain '
    'atau nested CV diperlukan sebelum menyimpulkan kemampuan generalisasi yang lebih luas.',
    '',
    '## 7. Artefak',
    '`svm_milk_quality.joblib`, `label_encoder.joblib`, `prediksi_test_svm.csv`, '
    '`prediksi_cv_oof_svm.csv`, `perbandingan_cv.csv`, `cv_results_tuning.csv`, '
    '`best_params.json`, `split_indices.json`, dan dua gambar confusion matrix.',
])
(OUTPUT_DIR/'documentation.md').write_text(summary + '\n', encoding='utf-8')

readme = '\n'.join([
    '# I-Smile OC26 — Klasifikasi Kualitas Susu dengan SVM',
    '',
    'Proyek klasifikasi 3 kelas (high/low/medium).',
    '**Isi nama anggota kelompok pada README repository utama.**',
    '',
    '## Dataset',
    'Milk Quality Dataset — https://www.kaggle.com/datasets/cpluzshrijayan/milkquality',
    'Simpan CSV sebagai `milknew.csv` di direktori notebook atau unggah saat menjalankan Google Colab.',
    '',
    '## Cara menjalankan',
    '1. Instal `numpy pandas matplotlib seaborn scikit-learn joblib notebook` bila belum tersedia.',
    '2. Buka notebook ini dan pilih **Run All**. Output otomatis dibuat dalam folder `output/`.',
    '',
    '## Ringkasan hasil terkini (dibuat otomatis dari run notebook)',
    f'- Split train {len(y_train)}, test {len(y_test)}; grup unik test {g_test.nunique()}.',
    f'- CV training: {n_splits} fold × {N_REPEATS} seed.',
    f'- Baseline F1-macro {cv_baseline_mean:.4f} ± {cv_baseline_std:.4f}.',
    f'- Tuned F1-macro {cv_tuned_mean:.4f} ± {cv_tuned_std:.4f}.',
    f'- Pilihan: {chosen_name} SVM.',
    f'- Holdout accuracy {holdout_metrics["Accuracy"]:.4f}, precision macro {holdout_metrics["Precision macro"]:.4f}, '
    f'recall macro {holdout_metrics["Recall macro"]:.4f}, F1 macro {holdout_metrics["F1 macro"]:.4f}.',
    '- Skor test sempurna, jika muncul, hanya berlaku untuk satu split dan bukan bukti performa sempurna pada semua data.',
    '',
    '## Penjelasan',
    'Lihat `documentation.md` untuk metode, metrik, interpretasi, visualisasi, dan keterbatasan.',
    'Gunakan README ini sebagai bahan untuk README repository GitHub dan lengkapi identitas anggota/kontribusi.',
])
(OUTPUT_DIR/'README.md').write_text(readme + '\n',encoding='utf-8')

print('Selesai. Artefak ditulis ke:', OUTPUT_DIR.resolve())
for f in sorted(OUTPUT_DIR.iterdir()):
    print(' -', f.name)
display(Markdown('### Kesimpulan akhir\n\n' +
                 f'Model final: **{chosen_name} SVM**. Perbandingan utama CV training = '
                 f'**{main_mean:.4f} ± {main_std:.4f}** ({n_splits} fold × {N_REPEATS} seed). '
                 f'Holdout Accuracy = **{holdout_metrics["Accuracy"]:.4f}**, dihitung pada '
                 f'**{test_total} sampel / {g_test.nunique()} kombinasi unik**. '
                 'Baca `output/documentation.md` untuk interpretasi lengkap.'))